In [66]:
!rm -rf dataset/*


In [39]:
# ---------- INSTALL ----------
!pip install -q tensorflow kaggle pillow numpy packaging>=24.1

# ---------- IMPORTS ----------
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from google.colab import files
import os, json, shutil, random

print("TensorFlow:", tf.__version__)

# ---------- KAGGLE AUTH ----------
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ---------- CLEAN & CREATE BASE FOLDERS ----------
!rm -rf healthcare_datasets dataset
!mkdir -p healthcare_datasets dataset

# 🧠 Brain Tumor MRI
!kaggle datasets download -d navoneel/brain-mri-images-for-brain-tumor-detection
!unzip -q brain-mri-images-for-brain-tumor-detection.zip -d healthcare_datasets/brain_mri

# 🎗️ Breast Cancer
!kaggle datasets download -d paultimothymooney/breast-histopathology-images
!unzip -q breast-histopathology-images.zip -d healthcare_datasets/breast_cancer

# 🫁 Chest X-ray
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database
!unzip -q covid19-radiography-database.zip -d healthcare_datasets/chest_xray

# 🧠 Alzheimer MRI
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset
!unzip -q augmented-alzheimer-mri-dataset.zip -d healthcare_datasets/alzheimer

# 🦠 Skin Cancer
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d healthcare_datasets/skin_cancer

print("✅ DATASETS READY")


TensorFlow: 2.19.0


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection
License(s): copyright-authors
brain-mri-images-for-brain-tumor-detection.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images
License(s): CC0-1.0
breast-histopathology-images.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
License(s): copyright-authors
covid19-radiography-database.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/uraninjo/augmented-alzheimer-mri-dataset
License(s): GNU Lesser General Public License 3.0
augmented-alzheimer-mri-dataset.zip: Skipping, found more recently modified local copy (use --force to force d

In [67]:
import os, shutil, random

SOURCE_BASE = "healthcare_datasets"
TARGET_BASE = "dataset"
os.makedirs(TARGET_BASE, exist_ok=True)

MAX_IMAGES = 300

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def collect_images(root):
    imgs = []
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                imgs.append(os.path.join(r, f))
    return imgs

# -------------------------------------------------
# 🧠 BRAIN MRI (YES / NO)
# -------------------------------------------------
for label, src in {
    "brain_tumor": "brain_mri/yes",
    "no_tumor": "brain_mri/no"
}.items():
    dst = os.path.join(TARGET_BASE, label)
    os.makedirs(dst, exist_ok=True)

    images = collect_images(os.path.join(SOURCE_BASE, src))
    random.shuffle(images)

    for img in images[:MAX_IMAGES]:
        shutil.copy(img, dst)

    print(f"{label}: {len(os.listdir(dst))} images")

# -------------------------------------------------
# 🎗️ BREAST CANCER (ROBUST – WORKING VERSION)
# -------------------------------------------------
def collect_breast_images(root):
    benign, malignant = [], []
    for r, _, files in os.walk(root):
        folder = os.path.basename(r)
        if folder == "0":   # benign
            for f in files:
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    benign.append(os.path.join(r, f))
        elif folder == "1": # malignant
            for f in files:
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    malignant.append(os.path.join(r, f))
    return benign, malignant

benign_imgs, malignant_imgs = collect_breast_images(
    os.path.join(SOURCE_BASE, "breast_cancer")
)

random.shuffle(benign_imgs)
random.shuffle(malignant_imgs)

for label, images in {
    "breast_benign": benign_imgs,
    "breast_malignant": malignant_imgs
}.items():
    dst = os.path.join(TARGET_BASE, label)
    os.makedirs(dst, exist_ok=True)

    for img in images[:MAX_IMAGES]:
        shutil.copy(img, dst)

    print(f"{label}: {len(os.listdir(dst))} images")

# -------------------------------------------------
# 🫁 CHEST X-RAY (FIXED)
# -------------------------------------------------
CHEST_BASE = "chest_xray/COVID-19_Radiography_Dataset"

for label, sub in {
    "covid": "COVID",
    "pneumonia": "Viral Pneumonia",
    "normal_xray": "Normal"
}.items():
    dst = os.path.join(TARGET_BASE, label)
    os.makedirs(dst, exist_ok=True)

    src = os.path.join(SOURCE_BASE, CHEST_BASE, sub)
    images = collect_images(src)
    random.shuffle(images)

    for img in images[:MAX_IMAGES]:
        shutil.copy(img, dst)

    print(f"{label}: {len(os.listdir(dst))} images")


## -------------------------------------------------
# 🧠 ALZHEIMER MRI (FIXED)
# -------------------------------------------------
ALZ_BASE = "alzheimer/AugmentedAlzheimerDataset"

for label, sub in {
    "alzheimer_mild": "MildDemented",
    "alzheimer_moderate": "ModerateDemented",
    "alzheimer_normal": "NonDemented"
}.items():
    dst = os.path.join(TARGET_BASE, label)
    os.makedirs(dst, exist_ok=True)

    src = os.path.join(SOURCE_BASE, ALZ_BASE, sub)
    images = collect_images(src)
    random.shuffle(images)

    for img in images[:MAX_IMAGES]:
        shutil.copy(img, dst)

    print(f"{label}: {len(os.listdir(dst))} images")

# -------------------------------------------------
# 🦠 SKIN CANCER (SAFE – UNIQUE FILENAMES)
# -------------------------------------------------
dst = os.path.join(TARGET_BASE, "skin_cancer")
os.makedirs(dst, exist_ok=True)

images = collect_images(os.path.join(SOURCE_BASE, "skin_cancer"))
random.shuffle(images)

for i, img in enumerate(images[:MAX_IMAGES]):
    ext = os.path.splitext(img)[1]
    new_name = f"skin_{i}{ext}"   # UNIQUE name
    shutil.copy(img, os.path.join(dst, new_name))

print(f"skin_cancer: {len(os.listdir(dst))} images")


print("\n✅ DATASET CREATION COMPLETE")


brain_tumor: 155 images
no_tumor: 98 images
breast_benign: 300 images
breast_malignant: 300 images
covid: 292 images
pneumonia: 282 images
normal_xray: 297 images
alzheimer_mild: 300 images
alzheimer_moderate: 300 images
alzheimer_normal: 300 images
skin_cancer: 300 images

✅ DATASET CREATION COMPLETE


In [68]:
for cls in sorted(os.listdir("dataset")):
    print(cls, len(os.listdir(os.path.join("dataset", cls))))


alzheimer_mild 300
alzheimer_moderate 300
alzheimer_normal 300
brain_tumor 155
breast_benign 300
breast_malignant 300
covid 292
no_tumor 98
normal_xray 297
pneumonia 282
skin_cancer 300


In [69]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train = datagen.flow_from_directory(
    "dataset",
    target_size=(224,224),
    batch_size=32,
    subset="training",
    class_mode="categorical",
    shuffle=True
)

val = datagen.flow_from_directory(
    "dataset",
    target_size=(224,224),
    batch_size=32,
    subset="validation",
    class_mode="categorical",
    shuffle=False
)


Found 2341 images belonging to 11 classes.
Found 583 images belonging to 11 classes.


In [ ]:
# ---------- MODEL ----------
model = models.Sequential([
    EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    ),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(train.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- TRAIN ----------
model.fit(train, validation_data=val, epochs=5)

# ---------- EXPORT ----------
model.export("saved_model")

with open("labels.json", "w") as f:
    json.dump(list(train.class_indices.keys()), f)

!zip -r saved_model.zip saved_model
files.download("saved_model.zip")
files.download("labels.json")

print("🎉 DONE – READY FOR FASTAPI")


Epoch 1/5
74/74 ━━━━━━━━━━━━━━━━━━━━ 951s 12s/step - accuracy: 0.5716 - loss: 1.0795 - val_accuracy: 0.7290 - val_loss: 0.8529
Epoch 2/5
11/74 ━━━━━━━━━━━━━━━━━━━━ 11:05 11s/step - accuracy: 0.8157 - loss: 0.4912